# RAG Experiments: two embedders x local Qwen3 generator (thinking off / on)

Extension of `rag_assignment.ipynb`. Same PDF, same chunks, same 8 test queries and the same cross-encoder, but:

* **Embedders compared:** `BAAI/bge-small-en-v1.5` (the main notebook's model) and `google/embeddinggemma-2` (float32).
* **Generator:** `Qwen/Qwen3-4B` run locally in float16, once with thinking **off** (greedy) and once with thinking **on** (sampled, as Qwen recommends).
* **Two gold labels fixed** compared with the main notebook (queries 2 and 8, see section 4).

GPU memory is only ~11 GB, so models are loaded one after another: embedders first (retrieval is computed and stored), then the LLM.

In [1]:
import os, re, time, json, gc, textwrap
import numpy as np, pandas as pd, faiss, pymupdf, torch
from sentence_transformers import SentenceTransformer, CrossEncoder

PDF = "/home2/akshat.pandey/rag/srd.pdf"
OUT = "/home2/akshat.pandey/rag/outputs_exp"
os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda"
LLM_NAME = "Qwen/Qwen3-4B"
print("gpu:", torch.cuda.get_device_name(0), "| free GB: %.1f" % (torch.cuda.mem_get_info()[0] / 1e9))

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()

gpu: NVIDIA GeForce RTX 2080 Ti | free GB: 11.2


## 1. Chunks (identical to the main notebook)

Two-column pages are read one column at a time; 200-word windows with 40-word overlap.

In [2]:
CHUNK_WORDS, OVERLAP = 200, 40
doc = pymupdf.open(PDF)
words, word_page = [], []
for pno, page in enumerate(doc, 1):
    r = page.rect
    top, bot, mid = r.y0 + 40, r.y1 - 40, (r.x0 + r.x1) / 2
    for col in (pymupdf.Rect(r.x0, top, mid, bot), pymupdf.Rect(mid, top, r.x1, bot)):
        for w in page.get_text("text", clip=col).split():
            words.append(w); word_page.append(pno)
chunks = [{"id": i, "page": word_page[s], "text": " ".join(words[s:s + CHUNK_WORDS])}
          for i, s in enumerate(range(0, len(words), CHUNK_WORDS - OVERLAP))]
print(len(chunks), "chunks")

1504 chunks


## 2. Test queries

Same eight queries as the main notebook. Two gold phrases were corrected:

* **Q2** used `can.t see you`, which matches the rule for a creature that cannot see the *attacker*. The right rule is "a target that you can't see".
* **Q8** used `speed becomes 0`, which matched the *Restrained* condition. It now requires the *Grappled* wording.

In [3]:
TESTS = [
 ("Keyword trap",          "What breaks my focus on a spell?",                          r"incapacitated or killed"),
 ("Negation / constraint", "Do I need to see a target to attack it?",                   r"target that you can.t see"),
 ("Exception / corner",    "Can I punish an unseen enemy that runs away from me?",      r"that you can see"),
 ("Negation / constraint", "Can I cast a spell while a silence effect is on me?",       r"can.t cast a spell with a verbal component"),
 ("Exception / corner",    "Can I shove a creature much bigger than me?",               r"no more than one size larger"),
 ("Keyword trap (synonym)","How many magic items can one creature bond with?",          r"no more than three magic items"),
 ("Counter-example",       "Can I rest twice in a single day to recover everything?",   r"more than one long rest"),
 ("Counter-example",       "Can a creature that is held by an enemy still walk away?",  r"grappled creature.s speed becomes 0"),
]

def gold_rank(order_ids, gold):
    g = re.compile(gold, re.I)
    for r, cid in enumerate(order_ids, 1):
        if g.search(chunks[cid]["text"]):
            return r, cid
    return None, None

def gold_chunks(gold):          # every chunk that contains the gold phrase
    g = re.compile(gold, re.I)
    return [c["id"] for c in chunks if g.search(c["text"])]

for cat, q, gold in TESTS:
    print(f"{q[:60]:<62} gold phrase found in chunks {gold_chunks(gold)}")

What breaks my focus on a spell?                               gold phrase found in chunks [372]
Do I need to see a target to attack it?                        gold phrase found in chunks [337]
Can I punish an unseen enemy that runs away from me?           gold phrase found in chunks [24, 25, 28, 40, 116, 118, 123, 170, 340, 395, 397, 411, 422, 423, 426, 428, 429, 430, 431, 436, 448, 451, 452, 453, 455, 456, 457, 458, 480, 483, 487, 493, 495, 496, 501, 508, 511, 512, 513, 514, 517, 524, 530, 531, 532, 543, 552, 553, 562, 563, 564, 565, 566, 568, 569, 570, 576, 583, 584, 586, 587, 593, 595, 597, 598, 602, 604, 605, 607, 616, 617, 623, 632, 633, 635, 661, 666, 699, 700, 702, 713, 735, 805, 806, 880, 881, 937]
Can I cast a spell while a silence effect is on me?            gold phrase found in chunks [369]
Can I shove a creature much bigger than me?                    gold phrase found in chunks [342, 343]
How many magic items can one creature bond with?               gold phrase found in 

## 3. Stage 1 and stage 2 for each embedder

For each embedder: embed all chunks, retrieve the top 10 for every test query, then re-rank those same 10 with the
cross-encoder. Only ids, ranks and scores are stored; the LLM step comes later.

In [4]:
EMBEDDERS = {
    "bge-small": dict(name="BAAI/bge-small-en-v1.5",
                      q_prefix="Represent this sentence for searching relevant passages: ", d_prefix="", kwargs={}),
    "embeddinggemma-2": dict(name="google/embeddinggemma-2",
                      q_prefix="task: search result | query: ", d_prefix="title: none | text: ",
                      kwargs={"torch_dtype": torch.float32}),     # card: bf16/fp32 only, fp16 can give NaN
}
ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)

def timeit(fn, reps=20):
    fn(); torch.cuda.synchronize()
    ts = []
    for _ in range(reps):
        t = time.perf_counter(); fn(); torch.cuda.synchronize(); ts.append(time.perf_counter() - t)
    return 1000 * float(np.mean(ts))

retrieval = {}      # retrieval[emb][query] = dict(...)
emb_info = {}
for key, cfg in EMBEDDERS.items():
    model = SentenceTransformer(cfg["name"], device=DEVICE, model_kwargs=cfg["kwargs"])
    t0 = time.perf_counter()
    E = model.encode([cfg["d_prefix"] + c["text"] for c in chunks], normalize_embeddings=True, batch_size=32)
    torch.cuda.synchronize(); t_index = time.perf_counter() - t0
    assert np.isfinite(E).all(), f"{key}: NaN/inf in embeddings"
    index = faiss.IndexFlatIP(E.shape[1]); index.add(E.astype("float32"))

    def stage1(q, k=10):
        qv = model.encode([cfg["q_prefix"] + q], normalize_embeddings=True).astype("float32")
        s, ids = index.search(qv, k)
        return [(int(i), float(x)) for i, x in zip(ids[0], s[0])]

    retrieval[key] = {}
    for cat, q, gold in TESTS:
        cands = stage1(q)
        ce_scores = ce.predict([(q, chunks[i]["text"]) for i, _ in cands], batch_size=10)
        order = [cands[j][0] for j in np.argsort(-ce_scores)]
        base_ids = [i for i, _ in cands]
        assert set(order) == set(base_ids)                  # re-ranking only re-orders the same 10
        gb, gcid = gold_rank(base_ids, gold); ga, _ = gold_rank(order, gold)
        retrieval[key][q] = dict(base_ids=base_ids, rerank_ids=order, gold_rank_base=gb, gold_rank_rerank=ga, gold_chunk=gcid,
                                 ce={int(cands[j][0]): float(ce_scores[j]) for j in range(10)})
    qs = [q for _, q, _ in TESTS]
    fixed = {q: stage1(q) for q in qs}                      # candidates fixed so stage 2 is timed on its own
    emb_info[key] = dict(dim=int(E.shape[1]), index_seconds=t_index,
                         stage1_ms=timeit(lambda: [stage1(q) for q in qs]) / len(qs),
                         stage2_ms=timeit(lambda: [ce.predict([(q, chunks[i]["text"]) for i, _ in fixed[q]], batch_size=10) for q in qs]) / len(qs))
    print(key, emb_info[key])
    del model, index, E; free_gpu()

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

bge-small {'dim': 384, 'index_seconds': 4.281103405868635, 'stage1_ms': 10.393493219453376, 'stage2_ms': 18.751800271274988}


Loading weights:   0%|          | 0/1376 [00:00<?, ?it/s]

embeddinggemma-2 {'dim': 768, 'index_seconds': 25.188630213029683, 'stage1_ms': 56.97263278707396, 'stage2_ms': 18.78164296358591}


## 4. Rank-shift comparison across embedders

Gold rank = position of the best-ranked chunk containing the answer phrase. `None` = not in the top 10.

In [5]:
rows = []
for cat, q, gold in TESTS:
    row = {"category": cat, "query": q}
    for key in EMBEDDERS:
        r = retrieval[key][q]
        row[f"{key}: base→rerank"] = f"{r['gold_rank_base']} → {r['gold_rank_rerank']}"
    rows.append(row)
rank_table = pd.DataFrame(rows)
pd.set_option("display.width", 250, "display.max_colwidth", 70)
rank_table.to_csv(f"{OUT}/rank_shift_by_embedder.csv", index=False)
rank_table

,category,query,bge-small: base→rerank,embeddinggemma-2: base→rerank
0,Keyword trap,What breaks my focus on a spell?,5 → 2,6 → 2
1,Negation / constraint,Do I need to see a target to attack it?,8 → 3,5 → 2
2,Exception / corner,Can I punish an unseen enemy that runs away from me?,4 → 3,5 → 5
3,Negation / constraint,Can I cast a spell while a silence effect is on me?,3 → 1,6 → 1
4,Exception / corner,Can I shove a creature much bigger than me?,3 → 1,None → None
5,Keyword trap (synonym),How many magic items can one creature bond with?,3 → 1,1 → 1
6,Counter-example,Can I rest twice in a single day to recover everything?,1 → 4,2 → 2
7,Counter-example,Can a creature that is held by an enemy still walk away?,None → None,None → None


## 5. Local LLM: Qwen3-4B, thinking off and on

* **Thinking off:** greedy decoding, up to 300 new tokens.
* **Thinking on:** sampling with Qwen's recommended settings (T=0.6, top-p 0.95, top-k 20) and a fixed seed, up to 3,000 new tokens.
  The text inside `<think>...</think>` is stored separately and removed from the answer. If the model never closes the
  think block within the limit, the answer is recorded as truncated.

The prompt is the same one used in the main notebook, so only the model changes.

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

PROMPT = """You are a rules assistant for Dungeons & Dragons 5th edition.
Answer the question using ONLY the context passages below. If the passages do not contain the answer, say
"The provided context does not answer this." Be concise (at most 4 sentences) and state any exception or restriction that applies.

Context:
{context}

Question: {question}
Answer:"""

def build_prompt(query, ids):
    ctx = "\n\n".join(f"[Passage {n+1}, page {chunks[i]['page']}] {chunks[i]['text']}" for n, i in enumerate(ids))
    return PROMPT.format(context=ctx, question=query)

tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16).to("cuda")   # plain load + move, no accelerate needed
llm.eval()
print("LLM loaded, free GB: %.1f" % (torch.cuda.mem_get_info()[0] / 1e9))

gen_cache = {}
def generate(prompt, thinking):
    key = (prompt, thinking)
    if key in gen_cache:
        return gen_cache[key]
    text = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False,
                                   add_generation_prompt=True, enable_thinking=thinking)
    inputs = tok([text], return_tensors="pt").to(0)
    kw = dict(do_sample=True, temperature=0.6, top_p=0.95, top_k=20, max_new_tokens=3000) if thinking \
         else dict(do_sample=False, max_new_tokens=300)
    torch.manual_seed(0)
    torch.cuda.synchronize(); t = time.perf_counter()
    with torch.no_grad():
        out = llm.generate(**inputs, **kw)
    torch.cuda.synchronize(); secs = time.perf_counter() - t
    new = out[0][inputs["input_ids"].shape[1]:]
    raw = tok.decode(new, skip_special_tokens=True)
    if "</think>" in raw:
        think, answer = raw.split("</think>", 1)
        truncated = False
    elif not thinking:
        think, answer, truncated = "", raw, False        # thinking off: the template already closed an empty think block
    else:
        think, answer, truncated = raw, "", True         # thinking on but no closing tag: ran out of tokens
    res = dict(answer=answer.strip(), think=think.replace("<think>", "").strip(), new_tokens=int(len(new)),
               seconds=secs, truncated=truncated)
    gen_cache[key] = res
    return res

# quick sanity check that fp16 works and the model follows the prompt
print(generate(build_prompt("What is 2+2?", [0]), False)["answer"][:200])

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

LLM loaded, free GB: 2.9


The provided context does not answer this.


## 6. Run every combination: embedder x pipeline x thinking mode

In [7]:
results = []
for key in EMBEDDERS:
    for cat, q, gold in TESTS:
        r = retrieval[key][q]
        for pipeline, ids in (("baseline", r["base_ids"][:3]), ("rerank", r["rerank_ids"][:3])):
            gold_in_ctx = any(re.search(gold, chunks[i]["text"], re.I) for i in ids)
            prompt = build_prompt(q, ids)
            for thinking in (False, True):
                g = generate(prompt, thinking)
                results.append(dict(embedder=key, category=cat, query=q, pipeline=pipeline, thinking=thinking,
                                    context_ids=ids, gold_in_context=gold_in_ctx, **g))
        print(key, "|", q[:50], "done", flush=True)
json.dump(dict(results=results, retrieval=retrieval, emb_info=emb_info, llm=LLM_NAME), open(f"{OUT}/experiments.json", "w"), indent=1, default=str)
print(len(results), "generations saved")

bge-small | What breaks my focus on a spell? done


bge-small | Do I need to see a target to attack it? done


bge-small | Can I punish an unseen enemy that runs away from m done


bge-small | Can I cast a spell while a silence effect is on me done


bge-small | Can I shove a creature much bigger than me? done


...

embeddinggemma-2 | Can I shove a creature much bigger than me? done


embeddinggemma-2 | How many magic items can one creature bond with? done


embeddinggemma-2 | Can I rest twice in a single day to recover everyt done


embeddinggemma-2 | Can a creature that is held by an enemy still walk done


64 generations saved


## 7. Answers side by side (no-think answers; thinking answers are in `experiments.json`)

In [8]:
df = pd.DataFrame(results)
for key in EMBEDDERS:
    print("#" * 100, "\nEMBEDDER:", key)
    for cat, q, gold in TESTS:
        print("-" * 100, f"\n[{cat}] {q}   gold rank {retrieval[key][q]['gold_rank_base']} → {retrieval[key][q]['gold_rank_rerank']}")
        for pipeline in ("baseline", "rerank"):
            for thinking in (False, True):
                x = df[(df.embedder == key) & (df["query"] == q) & (df.pipeline == pipeline) & (df.thinking == thinking)].iloc[0]
                tag = f"{pipeline:8s} think={'on ' if thinking else 'off'} ctx_has_gold={x.gold_in_context!s:5s} tok={x.new_tokens:<5d} {x.seconds:5.1f}s"
                print(tag, "\n    ", textwrap.fill(x.answer or "[no answer: thinking truncated]", 110, subsequent_indent="     "))

#################################################################################################### 
EMBEDDER: bge-small
---------------------------------------------------------------------------------------------------- 
[Keyword trap] What breaks my focus on a spell?   gold rank 5 → 2
baseline think=off ctx_has_gold=False tok=50      3.3s 
     Taking damage while concentrating on a spell requires a Constitution saving throw to maintain concentration.
     Casting another spell that requires concentration breaks your focus. Normal activity, such as moving and
     attacking, doesn’t interfere with concentration. The provided context does not answer this.
baseline think=on  ctx_has_gold=False tok=290    18.2s 
     The provided context states that concentration can be broken by casting another concentration-requiring spell
     or taking damage. Additionally, normal activity like moving or attacking doesn't interfere, but
     distractions (e.g., searching) may break concentration. 

## 8. Latency summary

In [9]:
lat = df.groupby(["thinking"]).agg(mean_seconds=("seconds", "mean"), mean_new_tokens=("new_tokens", "mean"),
                                   truncated=("truncated", "sum")).round(2)
print(lat)
print(pd.DataFrame(emb_info).T.round(2))
lat.to_csv(f"{OUT}/llm_latency.csv")
pd.DataFrame(emb_info).T.to_csv(f"{OUT}/embedder_latency.csv")

          mean_seconds  mean_new_tokens  truncated
thinking                                          
False             3.48            52.53          0
True             35.42           567.12          0
                    dim  index_seconds  stage1_ms  stage2_ms
bge-small         384.0           4.28      10.39      18.75
embeddinggemma-2  768.0          25.19      56.97      18.78
